# 02 — Animal Comparison

Expert (uniform) psychometric for an example WT vs HET, broken down by session / trial condition.

## Setup

In [ ]:
%matplotlib inline
from shared_setup import *
from analysis.phase import compute_phase

experiment, info = load_data()
print(f"Mode: {info['mode']}")

## Genotype groups

In [ ]:
# WT = light-delivery controls; HET = active-inhibition opto animals.
# SS01-13 are HETs that predate the opto cohort, so they're excluded from the opto HET group.
NON_OPTO_IDS = ['SS01', 'SS02', 'SS03', 'SS04', 'SS05', 'SS06',
                'SS07', 'SS08', 'SS09', 'SS10', 'SS11', 'SS12', 'SS13']

WT_IDS, HET_IDS = [], []
for animal_id in experiment.animal_ids:
    genotype = experiment.get_animal(animal_id).genotype.upper()
    if genotype == 'WT':
        WT_IDS.append(animal_id)
    elif genotype == 'HET' and animal_id not in NON_OPTO_IDS:
        HET_IDS.append(animal_id)

print(f"WT:  {WT_IDS}")
print(f"HET: {HET_IDS}")

In [ ]:
wt_animal = experiment.get_animal(WT_IDS[0])
het_animal = experiment.get_animal(HET_IDS[0])
print(f"WT  {wt_animal.animal_id} ({wt_animal.genotype})")
print(f"HET {het_animal.animal_id} ({het_animal.genotype})")

## Uniform (expert) — compute per condition

`compute_phase(animal, 'uniform', cohort='opto')` returns three dicts — `clean`, `psyc`, `um` —
keyed by the six condition labels (`baseline`, `masking`, `all_opto`, `opto_off`, `opto_on`,
`post_opto`). A condition with no surviving sessions is `None` in all three. `min_accuracy=0.6`
applies a session-quality floor to every condition.

In [ ]:
PRETTY = {
    'baseline':  'Last5 Sessions',
    'masking':   'Masking Sessions',
    'all_opto':  'Opto Sessions',
    'opto_off':  'Non-Opto Trials',
    'opto_on':   'Opto Trials',
    'post_opto': 'Post-Opto Trials',
}

clean_wt,  psyc_wt,  um_wt  = compute_phase(wt_animal,  'uniform', cohort='opto', min_accuracy=0.6)
clean_het, psyc_het, um_het = compute_phase(het_animal, 'uniform', cohort='opto', min_accuracy=0.6)

print(list(psyc_wt))

## Compare one condition: WT vs HET

Change `condition` to step through the six labels. A condition with no surviving sessions
for either animal is skipped rather than plotted.

In [ ]:
condition = 'opto_on'   # baseline | masking | all_opto | opto_off | opto_on | post_opto

if psyc_wt.get(condition) is None or psyc_het.get(condition) is None:
    missing = [g for g, p in [('WT', psyc_wt.get(condition)),
                              ('HET', psyc_het.get(condition))] if p is None]
    print(f"No surviving sessions for {condition!r}: {', '.join(missing)} — nothing to plot.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    plot_psychometric(psyc_wt[condition],  ax=axes[0])
    plot_psychometric(psyc_het[condition], ax=axes[1])
    axes[0].set_title(f"WT {wt_animal.animal_id} — {PRETTY[condition]}")
    axes[1].set_title(f"HET {het_animal.animal_id} — {PRETTY[condition]}")
    plt.tight_layout(); plt.show()

## Group averages — split by genotype

Group analog of the single-animal report. Per distribution, five views:
WT-vs-HET per condition; opto-vs-nonopto per genotype (the within-group laser
effect); mean update matrices; difference matrices (e.g. opto_on − opto_off) per
genotype; and summary-stat bars.

Two passes — **per-mouse average** (psychometric band = between-mouse SEM, honest;
fast) and **mega-pool** (band = trial-level bootstrap, descriptive only; the slow
pass, `POOLED_NBOOT` controls it). The helpers return their figures, so the same
figures are shown inline below and written to **one PDF per averaging method** in
the last cell.

Caveats kept on the figures: opto_on / post_opto matrices are low-trial and noisy
even averaged; difference matrices add the noise of both; mega-pool bands are not
testable error bars. Relies on `WT_IDS` / `HET_IDS` being populated.


In [ ]:
from analysis.phase import compute_group_phase, compute_group_stats
from behav_utils.plotting import plot_um
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display

DISTS       = {'uniform': 'Uniform', 'hard_a': 'Hard-A', 'hard_b': 'Hard-B'}
PANEL_ORDER = ['baseline', 'masking', 'all_opto', 'opto_off', 'opto_on', 'post_opto']
BAR_STATS   = ['accuracy', 'win_stay', 'lose_shift', 'recency']
CONTRAST_PAIRS = [('opto_on', 'opto_off'), ('opto_on', 'post_opto'), ('masking', 'baseline')]
GROUPS      = {g: ids for g, ids in [('WT', WT_IDS), ('HET', HET_IDS)] if ids}
GENO_COLOUR = {'WT': '#30638e', 'HET': '#d1495b'}
PAIR_COLOUR = ('#d62728', '#888888')      # a = effect (red), b = reference (grey)
MIN_ACC     = 0.6
POOLED_NBOOT = 200          # mega-pool band only; lower (e.g. 50) to iterate faster

print({g: len(ids) for g, ids in GROUPS.items()})
if not GROUPS:
    print("No genotype groups — check that genotype is populated (see NB30's GENOTYPE map).")


def _psych_by_condition(GP, phase, method, band_note):
    """WT vs HET group-average curves, one panel per condition."""
    conds = [c for c in PANEL_ORDER if any(GP[g][0].get(c) is not None for g in GP)]
    if not conds:
        return None
    ncol = 3; nrow = int(np.ceil(len(conds) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 4 * nrow), sharey=True)
    axes = np.atleast_1d(axes).flatten()
    for ax, cond in zip(axes, conds):
        for g in GP:
            r = GP[g][0].get(cond)
            if r is not None:
                plot_psychometric(r, ax=ax, color=GENO_COLOUR[g],
                                  label=f"{g} (n={len(GROUPS[g])})", show_ci=True, show_data=False)
        ax.set_title(PRETTY.get(cond, cond)); ax.legend(fontsize=8, frameon=False)
    for ax in axes[len(conds):]:
        ax.set_visible(False)
    fig.suptitle(f"{DISTS[phase]} — psychometric: WT vs HET per condition  "
                 f"({method}; band = {band_note})", y=1.01)
    fig.tight_layout()
    return fig


def _psych_within_genotype(GP, phase, method, band_note):
    """opto vs nonopto (and other pairs) per genotype — the within-group effect."""
    genos = list(GP)
    pairs = [(a, b) for a, b in CONTRAST_PAIRS
             if any(GP[g][0].get(a) is not None and GP[g][0].get(b) is not None for g in genos)]
    if not pairs:
        return None
    fig, axes = plt.subplots(len(genos), len(pairs),
                             figsize=(4.4 * len(pairs), 4 * len(genos)), squeeze=False, sharey=True)
    for i, g in enumerate(genos):
        for j, (a, b) in enumerate(pairs):
            ax = axes[i][j]; ra, rb = GP[g][0].get(a), GP[g][0].get(b)
            if ra is None or rb is None:
                ax.set_visible(False); continue
            plot_psychometric(rb, ax=ax, color=PAIR_COLOUR[1], label=PRETTY.get(b, b),
                              show_ci=True, show_data=False)
            plot_psychometric(ra, ax=ax, color=PAIR_COLOUR[0], label=PRETTY.get(a, a),
                              show_ci=True, show_data=False)
            ax.set_title(f"{g}: {PRETTY.get(a, a)} vs {PRETTY.get(b, b)}", fontsize=9)
            ax.legend(fontsize=7, frameon=False)
    fig.suptitle(f"{DISTS[phase]} — psychometric: opto vs nonopto per genotype  "
                 f"({method}; band = {band_note})", y=1.005)
    fig.tight_layout()
    return fig


def _um_by_condition(GP, phase, method):
    """Mean update matrices: conditions (rows) x genotype (cols)."""
    genos = list(GP)
    conds = [c for c in PANEL_ORDER if any(GP[g][1].get(c) is not None for g in genos)]
    if not conds:
        return None
    fig, axes = plt.subplots(len(conds), len(genos),
                             figsize=(3.6 * len(genos), 3.1 * len(conds)), squeeze=False)
    for i, cond in enumerate(conds):
        for j, g in enumerate(genos):
            ax = axes[i][j]; um = GP[g][1].get(cond)
            if um is None:
                ax.set_visible(False); continue
            plot_um(um, ax=ax, vmin=-0.4, vmax=0.4)
            ax.set_title(f"{g} — {PRETTY.get(cond, cond)}", fontsize=9)
    fig.suptitle(f"{DISTS[phase]} — update matrices ({method})", y=1.003)
    fig.tight_layout()
    return fig


def _um_difference(GP, phase, method):
    """Difference update matrices um[a]-um[b] per genotype (diverging, 0-centred)."""
    genos = list(GP)
    pairs = [(a, b) for a, b in CONTRAST_PAIRS
             if any(GP[g][1].get(a) is not None and GP[g][1].get(b) is not None for g in genos)]
    if not pairs:
        return None
    fig, axes = plt.subplots(len(genos), len(pairs),
                             figsize=(3.7 * len(pairs), 3.2 * len(genos)), squeeze=False)
    for i, g in enumerate(genos):
        for j, (a, b) in enumerate(pairs):
            ax = axes[i][j]; ua, ub = GP[g][1].get(a), GP[g][1].get(b)
            if ua is None or ub is None:
                ax.set_visible(False); continue
            diff = np.asarray(ua['um'], float) - np.asarray(ub['um'], float)
            vmax = float(np.nanmax(np.abs(diff))) or 1.0
            plot_um(diff, ax=ax, vmin=-0.2, vmax=0.2)
            ax.set_title(f"{g}: {PRETTY.get(a, a)} − {PRETTY.get(b, b)}", fontsize=8)
    fig.suptitle(f"{DISTS[phase]} — Δ update matrix ({method})",
                 y=1.003)
    fig.tight_layout()
    return fig


def _bar_panel(GS, phase, method):
    """One panel per stat; conditions on x, WT vs HET side by side."""
    genos = [g for g in GROUPS if (GS.genotype == g).any()]
    if not genos:
        return None
    x = np.arange(len(PANEL_ORDER)); w = 0.8 / len(genos)
    fig, axes = plt.subplots(1, len(BAR_STATS), figsize=(4.3 * len(BAR_STATS), 3.8))
    for ax, st in zip(np.atleast_1d(axes), BAR_STATS):
        for k, g in enumerate(genos):
            sub = GS[(GS.stat == st) & (GS.genotype == g)]
            xs = x + (k - (len(genos) - 1) / 2) * w
            if method == 'per_mouse' and 'animal' in GS.columns:
                for xi, p in zip(xs, PANEL_ORDER):
                    d = sub[sub.panel == p].value.dropna().to_numpy()
                    if not len(d):
                        continue
                    ax.scatter(np.full(len(d), xi), d, s=18, color=GENO_COLOUR[g], alpha=0.7, zorder=3)
                    ax.hlines(np.median(d), xi - w / 2.5, xi + w / 2.5, color=GENO_COLOUR[g], lw=2, zorder=4)
            else:
                vals = [sub[sub.panel == p].value.mean() if (sub.panel == p).any() else np.nan
                        for p in PANEL_ORDER]
                ax.bar(xs, vals, width=w, color=GENO_COLOUR[g], alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels([PRETTY.get(p, p) for p in PANEL_ORDER], rotation=45, ha='right', fontsize=8)
        ax.set_title(st); ax.spines[['top', 'right']].set_visible(False)
    handles = [plt.Line2D([0], [0], color=GENO_COLOUR[g], lw=3, label=g) for g in genos]
    fig.legend(handles=handles, loc='upper right', frameon=False)
    fig.suptitle(f"{DISTS[phase]} — summary stats, WT vs HET  ({method})", y=1.02)
    fig.tight_layout()
    return fig


def figures_for(phase, method, band_note):
    """Compute once per genotype, return the (non-empty) figures for one distribution."""
    GP = {g: compute_group_phase(experiment, phase, ids, method=method,
                                 min_accuracy=MIN_ACC, n_bootstrap=POOLED_NBOOT)
          for g, ids in GROUPS.items()}
    GS = pd.concat(
        [compute_group_stats(experiment, phase, ids, method=method, min_accuracy=MIN_ACC).assign(genotype=g)
         for g, ids in GROUPS.items()],
        ignore_index=True)
    figs = [
        _psych_by_condition(GP, phase, method, band_note),
        _psych_within_genotype(GP, phase, method, band_note),
        _um_by_condition(GP, phase, method),
        _um_difference(GP, phase, method),
        _bar_panel(GS, phase, method),
    ]
    return [f for f in figs if f is not None]


def show_distribution(phase, method, band_note):
    """Inline preview: display each figure then close it (no accumulation)."""
    for fig in figures_for(phase, method, band_note):
        display(fig); plt.close(fig)


PASSES = [('per_mouse', 'between-mouse SEM'), ('pooled', 'trial-level')]

### Pass 1 — per-mouse average  *(band = between-mouse SEM)*

In [ ]:
# for ph in DISTS:
#     show_distribution(ph, method='per_mouse', band_note='between-mouse SEM')

### Pass 2 — mega-pool  *(band = trial-level bootstrap, descriptive — slow cell)*

In [ ]:
# for ph in DISTS:
#     show_distribution(ph, method='pooled', band_note='trial-level (descriptive)')

### Save PDFs — one per averaging method

Writes `group_per_mouse_average.pdf` and `group_mega_pool.pdf`, each with one page
per figure across the three distributions. Re-runs the computation, so it is the
slow cell for the mega-pool pass — set `POOLED_NBOOT` low (and trim `DISTS`) while
drafting.


In [ ]:
from pathlib import Path

OUTDIR = Path('outputs/group_reports'); OUTDIR.mkdir(parents=True, exist_ok=True)
FILENAME = {'per_mouse': 'group_per_mouse_average.pdf', 'pooled': 'group_mega_pool.pdf'}

for method, band_note in PASSES:
    path = OUTDIR / FILENAME[method]
    n = 0
    with PdfPages(path) as pdf:
        for ph in DISTS:
            for fig in figures_for(ph, method, band_note):
                pdf.savefig(fig, bbox_inches='tight'); plt.close(fig); n += 1
    print(f"{n} pages -> {path}")